In [1]:
import numpy as np
import pandas as pd


In [2]:
# Define your files 
files = {}

In [3]:
def parse_hmm_transitions(hmm_file):

    """ 
    Extract transition probabilities from a HMMER3 profile.

    Returns:
        NumPy array containing normalised:
        [M→M, M→I, M→D]
        transition probabilities for each node.
    """

    transitions = []
    in_main_model = False

    with open(hmm_file) as f:
        for line in f:
            line = line.strip()
            
            # Start parsing after the COMPO line (the end of the header)
            if line.startswith("COMPO"):
                in_main_model = True
                continue
            if line == "//":
                in_main_model = False
            
            if not in_main_model:
                continue

            parts = line.split()
            
            # In HMMER3, transition lines usually follow the match and insert emission lines.
            # They are the only lines in the main body with 7 numerical/asterisk values.
            if len(parts) == 7:
                try:
                    # Convert '*' to a very large number so exp(-val) becomes 0
                    processed_parts = [np.inf if x == '*' else float(x) for x in parts]
                    vals = np.array(processed_parts)

                    # HMMER values are -ln(P). To get P: P = exp(-val)
                    probs = np.exp(-vals)

                    # Normalizing the first three: M->M, M->I, M->D
                    # These three must sum to 1.0 at every node
                    m_trans = probs[:3]
                    row_sum = m_trans.sum()
                    
                    if row_sum > 0:
                        transitions.append(m_trans / row_sum)

                except ValueError:
                    continue

    return np.array(transitions)

In [4]:
def compute_transition_stats(hmm_file):
    
    # Extract normalised transition probabilities from the HMM profile
    # Shape: (positions, 3)
    # Columns = [M→M, M→I, M→D]
    
    transitions = parse_hmm_transitions(hmm_file)
    
    # Calculate the mean probability of each transition type
    return {
        "M->M_mean": transitions[:, 0].mean(),
        "M->I_mean": transitions[:, 1].mean(),
        "M->D_mean": transitions[:, 2].mean()
    }


# Run transition analysis for every HMM profile in the files dictionary
transition_stats = {
    name: compute_transition_stats(path)
    for name, path in files.items()
}



In [5]:
transition_df = pd.DataFrame(transition_stats).T

print("\nTransition probabilities:")
print(transition_df)


Transition probabilities:
      M->M_mean  M->I_mean  M->D_mean
BfrA   0.944196   0.029413   0.026392
BfrB   0.968985   0.018609   0.012406
Ftn    0.977495   0.013866   0.008638


In [6]:
# Create a copy for reporting
transition_pct = transition_df * 100

# Rename columns to reflect the change
transition_pct.columns = ["M->M (%)", "M->I (%)", "M->D (%)"]

print("Transition Probabilities (as %):")
print(transition_pct.round(2)) # Rounded to 2 decimal places for clarity

Transition Probabilities (as %):
      M->M (%)  M->I (%)  M->D (%)
BfrA     94.42      2.94      2.64
BfrB     96.90      1.86      1.24
Ftn      97.75      1.39      0.86


In [7]:
def parse_hmm_emissions_corrected(hmm_file):

    """
    Parses HMMER3 profile HMM emission probabilities for amino acid match states.

    Reads a HMMER3 .hmm file and extracts match-state emission scores
    for the 20 standard amino acids. Emissions are converted back into probabilities using
    P = 2^(-score/1000). Impossible emissions ('*') are treated as zero probability.

    Each HMM position is converted to a normalised probability distribution over
    amino acids, ensuring each row sums to 1. The output is returned as a
    DataFrame with amino acids as columns and HMM positions as rows.
    """

    
    aa_order = list("ACDEFGHIKLMNPQRSTVWY")
    emissions = []

    with open(hmm_file) as f:
        # Flag to ensure we are inside the model body
        in_main_model = False
        
        for line in f:
            line = line.strip()
            
            # The main model starts after the 'HMM' header line
            if line.startswith("HMM"):
                in_main_model = True
                continue
            
            if not in_main_model or not line:
                continue

            if line == "//":
                break

            parts = line.split()

            # HMMER3 specific: Match emission lines start with the node index (1, 2, 3...)
            # They contain the node index plus 20 amino acid scores.
            if parts[0].isdigit() and len(parts) >= 21:
                try:
                    # Skip the first part (the index) and take the next 20 values
                    raw_vals = parts[1:21]
                    
                    probs = []
                    for val in raw_vals:
                        if val == "*":
                            probs.append(0.0)
                        else:
                            # HMMER log-odds are base 2 and scaled by 1000
                            # P = 2^(-score/1000)
                            probs.append(2 ** (-float(val) / 1000))
                    
                    probs = np.array(probs, dtype=float)

                    # Normalize to ensure it's a true probability distribution
                    total = probs.sum()
                    if total > 0:
                        probs /= total
                    
                    emissions.append(probs)
                except ValueError:
                    # Skips lines that aren't purely numerical match emissions
                    continue

    if not emissions:
        raise ValueError(f"No valid emission rows found in {hmm_file}. Check format.")

    return pd.DataFrame(emissions, columns=aa_order)

# Re-run your alignment and matrix logic with this corrected function
dfs = {name: parse_hmm_emissions_corrected(path) for name, path in files.items()}


In [8]:
def align_profiles(dfs_dict):
    """
    Align multiple HMM emission df to a shared positional length.

    Standardises a set of HMM-derived emission matrices by
    truncating all profiles to the length of the shortest profile. This ensures
    positional comparability across all profiles.

    Parameters
    ----------
    dfs_dict : dict
        Dictionary of {profile_name: pandas.DataFrame}, where each DataFrame
        contains HMM emission probabilities indexed by position.

    Returns
    -------
    dict
        Dictionary of aligned DataFrames with identical number of rows.
    """

    if not dfs_dict:
        raise ValueError("Input dictionary is empty. Provide at least one DataFrame.")

    # Find shortest HMM profile length
    min_len = min(df.shape[0] for df in dfs_dict.values())

    # Truncate all profiles to the same length
    aligned = {
        name: df.iloc[:min_len].reset_index(drop=True)
        for name, df in dfs_dict.items()
    }

    return aligned

In [9]:
# =========================================================
# 1. Parse ALL HMM profiles into emission + transition sets
# =========================================================

dfs_emissions_raw = {
    name: parse_hmm_emissions_corrected(path)
    for name, path in files.items()
}

dfs_transitions_raw = {
    name: pd.DataFrame(
        parse_hmm_transitions(path),
        columns=["M->M", "M->I", "M->D"]
    )
    for name, path in files.items()
}


# =========================================================
# 2. Align all profiles to shared positional coordinate system
#    (based on shortest profile length)
# =========================================================

dfs_emiss_aligned = align_profiles(dfs_emissions_raw)
dfs_trans_aligned = align_profiles(dfs_transitions_raw)

In [10]:
def compute_combined_distance(df_emiss_i, df_emiss_j, df_trans_i, df_trans_j):
    """
    Compute a combined distance between two HMM profiles.

    The distance is defined as the average of:
    1. Mean absolute difference between amino acid emission distributions
    2. Mean absolute difference between transition probability distributions

    Both are computed across all aligned HMM positions.

    Returns
    -------
    float
        Combined distance score (lower = more similar profiles).
    """

    # ---------------------------------------------------------
    # Emission distance (sequence-level signal)
    # shape: (positions × 20 amino acids)
    # ---------------------------------------------------------
    seq_dist = (df_emiss_i - df_emiss_j).abs().to_numpy().mean()

    # ---------------------------------------------------------
    # Transition distance (structural / state-level signal)
    # shape: (positions × 3 transitions)
    # ---------------------------------------------------------
    struct_dist = (df_trans_i - df_trans_j).abs().to_numpy().mean()

    # ---------------------------------------------------------
    # Combine both components equally
    # ---------------------------------------------------------
    return (seq_dist + struct_dist) / 2

Combined Distance Matrix:
          BfrA      BfrB       Ftn
BfrA  0.000000  0.018657  0.019288
BfrB  0.018657  0.000000  0.010078
Ftn   0.019288  0.010078  0.000000


In [ ]:
# Get profile names dynamically (no hardcoding)
names = list(dfs_emiss_aligned.keys())

# Initialise empty square distance matrix
combined_matrix = pd.DataFrame(index=names, columns=names, dtype=float)

# Compute pairwise distances
for i in names:
    for j in names:

        combined_matrix.loc[i, j] = compute_combined_distance(
            dfs_emiss_aligned[i],
            dfs_emiss_aligned[j],
            dfs_trans_aligned[i],
            dfs_trans_aligned[j]
        )

# Optional: set diagonal explicitly to 0 (self-distance)
for name in names:
    combined_matrix.loc[name, name] = 0.0


print("Combined Distance Matrix:")
print(combined_matrix)